# 14-amaliyot — Sodda RAG tizimini qurish (m14)

**Mavzu:** Bilim bazasidan qidirib, javobni kontekstga tayantiramiz (Retrieval-Augmented Generation).
**Kuni:** 15-kun (Day 15) · **Juft ma'ruza:** [L14 — RAG](../lectures/d14_rag.pdf)
**Quriladigan kapstone moduli:** `m14 RAGEngine` (haqiqiy modul — `consumed_by`: m15 agent `retrieve_docs`).

Bugun L14 da o'rgangan **RAG jarayoni**, **vektor qidiruv** va **kontekstli javob** ni amalda quramiz.

**Bugungi maqsadlar (L14 [C] dan):**
1. RAG prompt token byudjetini hisoblaymiz (L14 [I2]: 3 chunk + instruction = 700 token, <1%).
2. Hujjatlarni chunk + embedding qilib indekslaymiz; vektor qidiruv (top-k) bilan topamiz.
3. `m14 RAGEngine` ga ulab, `answer()` orqali javob + sources + confidence olamiz.

> ⚠️ **Korpus litsenziyasi:** uz_kb (yangiliklar + lex.uz). Offline rejimda kichik original uz bilim bazasi.
> **Offline:** sentence-transformers/FAISS/LLM o'rniga TF-IDF (char n-gram) + numpy kosinus + ekstraktiv javob.

## 1. Muhit tekshiruvi

Kutubxonalar, urug' va bayroqlar. `sentence-transformers`/`faiss`/LLM API bo'lmasa ham notebook
uchdan-uchgacha ishlaydi: offline rejimda **TF-IDF (char n-gram) + numpy kosinus + ekstraktiv** javob.

In [ ]:
import os, sys, random, warnings
warnings.filterwarnings("ignore")
import numpy as np

OFFLINE_FALLBACK = True          # mahalliy: model yuklab olish / faiss / LLM API yo'q
random.seed(42); np.random.seed(42)

try:
    import sentence_transformers   # noqa: F401
    HAS_ST = True
except Exception:                   # ImportError yoki torch DLL xatosi -> fallback
    HAS_ST = False

try:
    import faiss                    # noqa: F401
    HAS_FAISS = True
except Exception:
    HAS_FAISS = False

# Kaggle (GPU+internet): OFFLINE_FALLBACK=False -> ST embedding + FAISS + LLM API.
# Mahalliy: TF-IDF + numpy kosinus + ekstraktiv javob (tez, internetsiz).
USE_ST = HAS_ST and not OFFLINE_FALLBACK
USE_LLM = False                     # LLM API kaliti + internet (Kaggle); offline ekstraktiv

print("sentence-transformers:", ("bor" if HAS_ST else "yo'q"),
      "| faiss:", ("bor" if HAS_FAISS else "yo'q"))
print("OFFLINE_FALLBACK =", OFFLINE_FALLBACK, "| USE_ST =", USE_ST, "| USE_LLM =", USE_LLM)

for _cand in ["../../capstone/modules", "/kaggle/working/capstone/modules", "capstone/modules"]:
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break

CKPT = "d15_checkpoints"
CORPUS_PATH = os.path.join(CKPT, "uz_kb_mini.txt")

## 2. Yaxlit natija (avval manzil)

Tugallangan natijani ko'ramiz: bilim bazasini indekslab, bir savolga RAG javobi + manbalar + ishonch.

In [ ]:
import m14_rag_engine as M14
from m14_rag_engine import RAGEngine
M14.USE_ST = USE_ST          # offline -> False (TF-IDF)
M14.USE_LLM = USE_LLM        # offline -> False (ekstraktiv)

def load_kb(path):
    """Bilim bazasini o'qiydi (har qator -- bitta hujjat/chunk)."""
    with open(path, encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]

kb = load_kb(CORPUS_PATH)
print("Bilim bazasi:", len(kb), "hujjat")

rag = RAGEngine()
rag.index(kb)
res = rag.answer("O'zbekiston poytaxti qaysi shahar?", k=3)
print("\nJavob   :", res["answer"][:90], "...")
print("Manbalar:", len(res["sources"]), "ta")
print("Ishonch :", round(res["confidence"], 3))

## 3. Tayyor kod bloki — periferiya (PRIMM)

Bu yerda **to'liq berilgan** kod: Kaggle yo'lida embedding (sentence-transformers), vektor DB (FAISS
`IndexFlatIP`) va LLM API ulanishi. Offline rejimda ularning o'rniga TF-IDF + numpy kosinus ishlaydi.

### Bashorat qiling
Offline rejimda embedding sifatida nima ishlatiladi (kod `USE_ST` False bo'lganda)? Bu o'zbek
qo'shimchalarini (masalan "poytaxt" va "poytaxti") qanday hisobga oladi?

In [ ]:
# Kaggle yo'li (USE_ST=True, internet) -- ko'rsatiladi, mahalliyda bajarilmaydi:
#   from sentence_transformers import SentenceTransformer
#   model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#   emb = model.encode(texts)               # (n, 384)
#   import faiss
#   index = faiss.IndexFlatIP(emb.shape[1]) # ichki ko'paytma (kosinus, normallangan)
#   index.add(emb)
#   # LLM API: from openai import OpenAI; client.chat.completions.create(...)

# Offline yo'li (mahalliy): TF-IDF char n-gram -- qo'shimchalarni n-gram orqali bog'laydi
from sklearn.feature_extraction.text import TfidfVectorizer
demo_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
X = demo_vec.fit_transform(["poytaxt", "poytaxti", "samarqand"]).toarray()
sim = X @ X.T
print("poytaxt vs poytaxti kosinus:", round(float(sim[0, 1]), 3))
print("poytaxt vs samarqand kosinus:", round(float(sim[0, 2]), 3))

### Tekshiring
1. "poytaxt" va "poytaxti" o'xshashligi "samarqand" bilan o'xshashlikdan katta-mi? (char n-gram ulashadi.)
2. Kaggle yo'lida FAISS `IndexFlatIP` nega ichki ko'paytma (inner product) ishlatadi? (normallangan = kosinus.)
3. LLM API bo'lmasa, offline javob qanday hosil bo'ladi? (ekstraktiv -- top-k manbalar birlashtiriladi.)

### O'zgartiring
`demo_vec` ni so'z-darajali (`analyzer="word"`) ga o'zgartiring va "poytaxt"/"poytaxti" o'xshashligi
0 ga tushganini ko'ring -- o'zbek morfologiyasi muammosi.

### ✅ Checkpoint A
Bilim bazasini qaytadan yuklaydi — oldingi kataklar ishlamagan bo'lsa, shu yerdan davom etamiz.

In [ ]:
if OFFLINE_FALLBACK or "kb" not in dir():
    kb = load_kb(CORPUS_PATH)
print("Checkpoint A:", len(kb), "hujjat tayyor.")

## 4. Asosiy mavzu — so'nuvchi tayanch

Avval L14 [I2] dagi prompt token byudjetini tekshiramiz, so'ng chunking + indekslash + qidiruvni
birgalikda quramiz, oxirida `m14` orqali javob olamiz.

### 4A. Namuna (men qilaman): RAG prompt token byudjeti

L14 [I2]: 3 ta chunk (≈200 token) + instruction (≈100 token) = 700 token; kontekst oynasining 1% idan kam.

In [ ]:
def prompt_tokens(k, t_chunk, t_instr):
    """RAG prompt token soni = k * chunk + instruction (L14 [G2])."""
    return k * t_chunk + t_instr

T = prompt_tokens(3, 200, 100)
window = 128000
print("prompt =", T, "token | ulush =", round(T / window, 4))
assert T == 700                      # Ma'ruza L14 [I2]-slayd bilan solishtiring
assert T / window < 0.01
print("To'g'ri! RAG prompt = 700 token (<1% oyna) -- L14 [I2] bilan mos.")

### 4B. Birgalikda (biz qilamiz): hujjat chunking

Uzun hujjatni ustma-ust (overlap) bo'laklarga ajratamiz. (Kaggle: `RecursiveCharacterTextSplitter`;
bu yerda sodda char-oynali variant.)

In [ ]:
def chunk_text(text, size=200, overlap=20):
    """Matnni overlap bilan char-oynali bo'laklarga ajratadi."""
    chunks = []
    start = 0
    step = size - overlap
    # === SIZNING KODINGIZ (taxminan 3 qator) ===
    # while start < len(text): bo'lak text[start:start+size] ni qo'shing,
    # start ni step ga oshiring
    raise NotImplementedError("chunk_text siklini to'ldiring")
    return chunks

In [ ]:
parchalar = chunk_text("a" * 500, size=200, overlap=20)
print("bo'laklar soni:", len(parchalar), "| uzunliklari:", [len(c) for c in parchalar])
assert len(parchalar) == 3
assert all(len(c) <= 200 for c in parchalar)
print("To'g'ri! 500 belgi -> 3 ta overlapli bo'lak.")

### 4C. Birgalikda: indekslash va qidiruv

`m14 RAGEngine` ni bilim bazasida indekslang va savolga javob oling (top-k retrieve).

In [ ]:
eng = RAGEngine()
# === SIZNING KODINGIZ (2 qator) ===
# eng ni kb bilan index qiling, so'ng "Mehnat shartnomasi nima?" savoliga k=3 bilan javob oling
raise NotImplementedError("index() va answer() ni to'ldiring")
print("Manbalar soni:", len(javob["sources"]))
print("Ishonch:", round(javob["confidence"], 3))

In [ ]:
assert set(javob.keys()) == {"answer", "sources", "confidence"}
assert isinstance(javob["answer"], str)
assert len(javob["sources"]) == 3          # retrieved_docs = 3 (L14 [I2])
assert 0.0 <= javob["confidence"] <= 1.0
print("To'g'ri! answer -> dict {answer, sources, confidence}; sources = 3.")

### 4D. Mustaqil (siz qilasiz): o'z savolingiz

Boshqa savol bering va javob tuzilishini tekshiring. (Offline javob ekstraktiv -- top-k manbalar
birlashtiriladi; sifat demo-darajada, bu halol.)

In [ ]:
mening = eng.answer("Soliq kim tomonidan to'lanadi?", k=2)
print("Javob:", mening["answer"][:90], "...")
print("Manbalar:", len(mening["sources"]), "| ishonch:", round(mening["confidence"], 3))

assert set(mening.keys()) == {"answer", "sources", "confidence"}
assert len(mening["sources"]) == 2
assert all(isinstance(s, str) for s in mening["sources"])
print("To'g'ri! k=2 da 2 ta manba, dict tuzilishi saqlandi.")

## 5. Loyihaga ulash — `m14 RAGEngine`

Bugungi kod `capstone/modules/m14_rag_engine.py` ga yig'ilgan (interfeys `contracts.py` dan). U
**haqiqiy modul**: `save_index`/`load_index` bor va m15 agent (`retrieve_docs`) uni ishlatadi.

In [ ]:
import tempfile
api = RAGEngine()
for m in ("index", "answer", "save_index", "load_index"):
    assert hasattr(api, m), "yetishmayotgan metod: " + m

api.index(kb)
_p = os.path.join(tempfile.gettempdir(), "m14_index.pkl")
api.save_index(_p)
api2 = RAGEngine()
api2.load_index(_p)
qayta = api2.answer("Poytaxt qaysi shahar?", k=3)
print("Yuklangan indeksdan manbalar:", len(qayta["sources"]))
assert set(qayta.keys()) == {"answer", "sources", "confidence"}
assert len(qayta["sources"]) == 3
os.remove(_p)
print("To'g'ri! m14 shartnomaga mos; save_index/load_index ishlaydi.")

**Git (o'z repozitoriyangizda):**
```bash
git add capstone/modules/m14_rag_engine.py
git commit -m "P14: m14 RAGEngine qo'shildi"
git push
```

## 6. Tadqiqot savoli + yakun

**Tajriba:** `k` ni 1, 3, 5 qilib o'zgartiring va manbalar soni hamda ishonchni kuzating.

In [ ]:
for k in (1, 3, 5):
    r = eng.answer("O'zbekiston daryolari qaysi?", k=k)
    print("k =", k, "-> manbalar:", len(r["sources"]), "| ishonch:", round(r["confidence"], 3))

**O'ylab ko'ring:** chunking hajmi (size) qidiruv sifatiga qanday ta'sir qiladi -- juda katta yoki
juda kichik bo'lak nega yomon? `k` ni oshirsak, prompt token byudjeti qanday o'zgaradi (L14 [I2])?
O'zbek qo'shimchalari uchun char n-gram embedding nega so'z-darajalidan yaxshiroq?

---
**Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima? (Bir jumla.)